# Claim Occurrence EDA

## Objective

The goal of this notebook is to understand the data before building a claim occurrence model.

The model will predict whether an Auto insurance contract has a claim during its contract period.

The dataset comes from the `vw_auto_claim_occurrence_ml` SQL view.

Each row represents one fully observed Auto contract with vehicle information.

The target variable is:

- `1`: A valid claim occurred during the contract period.
- `0`: No valid claim occurred during the contract period.

The EDA will focus on data quality, feature distributions, target relationships, and the final feature engineering strategy.

In [3]:
## Setup

import pyodbc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")


pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [4]:
conn = pyodbc.connect(
    "DSN=InsuranceAnalytics;",
    autocommit=True
)

query = """
SELECT *
FROM vw_auto_claim_occurrence_ml
"""

df = pd.read_sql(query, conn)

print("Shape:", df.shape)
print("Unique contracts:", df["contract_id"].nunique())
print("Claims:", df["has_claim"].sum())
print(
    "Claim rate:",
    
    f"{df['has_claim'].mean() * 100:.2f}%"
)

Shape: (4327, 20)
Unique contracts: 4327
Claims: 88
Claim rate: 2.03%


## Data Quality Check

Before the analysis, we check the quality of the modeling dataset.

We will check:

- duplicate contracts,
- missing values,
- data types,
- target distribution,
- invalid numeric values.

Each row should represent one unique Auto insurance contract.

In [5]:
print("Dataset shape:", df.shape)

print(
    "Duplicate rows:",
    df.duplicated().sum()
)

print(
    "Duplicate contract IDs:",
    df["contract_id"].duplicated().sum()
)

df.info()

Dataset shape: (4327, 20)
Duplicate rows: 0
Duplicate contract IDs: 0
<class 'pandas.DataFrame'>
RangeIndex: 4327 entries, 0 to 4326
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   contract_id      4327 non-null   str    
 1   start_date       4327 non-null   object 
 2   end_date         4327 non-null   object 
 3   annual_premium   4327 non-null   float64
 4   city             4327 non-null   str    
 5   risk_zone        4327 non-null   str    
 6   client_age       3964 non-null   float64
 7   channel          4327 non-null   str    
 8   csp              3820 non-null   str    
 9   gender           3488 non-null   str    
 10  brand            4327 non-null   str    
 11  model            4327 non-null   str    
 12  year             4110 non-null   float64
 13  power_hp         3615 non-null   float64
 14  fuel_type        4327 non-null   str    
 15  current_value    4327 non-null   float64
 16  c

In [6]:
missing_summary = (
    df.isna().sum().to_frame("missing_count")
)

missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100)

missing_summary = (missing_summary.loc[missing_summary["missing_count"] > 0]
                   .sort_values("missing_pct", ascending=False))

missing_summary

,missing_count,missing_pct
gender,839,19.3899
power_hp,712,16.4548
color,673,15.5535
csp,507,11.7171
previous_claims,433,10.0069
client_age,363,8.3892
year,217,5.0150


In [7]:
cardinality_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "unique_values": df.nunique(dropna=True),
        "missing_count": df.isna().sum()
    }
).sort_values("unique_values")

cardinality_summary

,dtype,unique_values,missing_count
gender,str,2,839
has_claim,int64,2,0
vehicle_usage,str,3,0
risk_zone,str,3,0
fuel_type,str,4,0
previous_claims,float64,4,433
channel,str,4,0
brand,str,5,0
color,str,5,673
csp,str,7,507


In [8]:
target_summary = (
    df["has_claim"]
    .value_counts()
    .rename_axis("has_claim")
    .reset_index(name="contracts")
)

target_summary["percentage"] = (
    target_summary["contracts"]
    / len(df)
    * 100
)

target_summary

,has_claim,contracts,percentage
0,0,4239,97.9663
1,1,88,2.0337


In [9]:
df["start_date"] = pd.to_datetime(df["start_date"])
df["end_date"] = pd.to_datetime(df["end_date"])

print(df[["start_date", "end_date"]].dtypes)

start_date    datetime64[s]
end_date      datetime64[s]
dtype: object


### Data Quality Findings

The dataset contains 4,327 unique Auto contracts and no duplicate records.

There are 88 contracts with a claim, which gives a positive rate of about 2.03%. The target is highly imbalanced.

Some customer and vehicle features contain missing values. However, every contract in this dataset has a vehicle record.

The next step is to check numeric ranges and derived vehicle age.

In [10]:
quality_checks = pd.Series(
    {
        "premium_le_zero": (
            df["annual_premium"] <= 0
        ).sum(),

        "client_age_below_18": (
            df["client_age"] < 18
        ).sum(),

        "client_age_above_100": (
            df["client_age"] > 100
        ).sum(),

        "power_hp_le_zero": (
            df["power_hp"] <= 0
        ).sum(),

        "current_value_le_zero": (
            df["current_value"] <= 0
        ).sum(),

        "previous_claims_below_zero": (
            df["previous_claims"] < 0
        ).sum(),

        "end_before_start": (
            df["end_date"] < df["start_date"]
        ).sum(),
    }
)

quality_checks

premium_le_zero               0
client_age_below_18           0
client_age_above_100          0
power_hp_le_zero              0
current_value_le_zero         0
previous_claims_below_zero    0
end_before_start              0
dtype: int64

In [11]:
df["vehicle_age"] = (
    df["start_date"].dt.year
    - df["year"]
)

vehicle_age_check = pd.Series(
    {
        "missing_vehicle_age": (
            df["vehicle_age"].isna().sum()
        ),
        "negative_vehicle_age": (
            df["vehicle_age"] < 0
        ).sum(),
        "vehicle_age_above_50": (
            df["vehicle_age"] > 50
        ).sum(),
    }
)

vehicle_age_check

missing_vehicle_age     217
negative_vehicle_age    191
vehicle_age_above_50      0
dtype: int64

### Data Quality Results

The numeric variables are within reasonable ranges.

No invalid values were found for premium, client age, vehicle power, vehicle value, previous claims, or contract dates.

Some vehicle ages are negative because the vehicle model year is later than the contract start year. These values will be changed to zero during feature engineering.

Missing vehicle ages will be handled during preprocessing.

In [12]:
numeric_features = [
    "annual_premium",
    "client_age",
    "power_hp",
    "current_value",
    "previous_claims",
    "vehicle_age",
]

numeric_summary = df[numeric_features].describe().T

numeric_summary["missing_pct"] = (
    df[numeric_features].isna().mean() * 100
)

numeric_summary["skewness"] = (
    df[numeric_features].skew()
)

numeric_summary

,count,mean,std,min,25%,50%,75%,max,missing_pct,skewness
annual_premium,"4,327.0000",851.2803,248.8527,448.3000,656.3050,804.0000,"1,007.1150","1,820.0600",0.0000,0.6941
client_age,"3,964.0000",43.6365,12.2563,18.0000,35.0000,44.0000,52.0000,75.0000,8.3892,0.0282
power_hp,"3,615.0000",144.8217,31.7195,88.5100,117.3700,144.8300,172.9900,200.0000,16.4548,-0.0116
current_value,"4,327.0000","10,193.3623","6,617.8814","4,000.1000","5,380.9800","8,366.1600","12,856.6000","47,482.8000",0.0000,2.0155
previous_claims,"3,894.0000",1.4941,1.1168,0.0000,1.0000,1.0000,3.0000,3.0000,10.0069,0.0209
vehicle_age,"4,110.0000",4.5311,3.8783,-2.0000,2.0000,4.0000,6.0000,15.0000,5.0150,0.7348


In [13]:
def categorical_claim_rate(df, feature):
    result = (
        df.assign(
            category=df[feature].fillna("Missing")
        )
        .groupby("category")
        .agg(
            contracts=("contract_id", "size"),
            claims=("has_claim", "sum"),
            claim_rate=("has_claim", "mean"),
        )
        .reset_index()
    )

    result["claim_rate_pct"] = (
        result["claim_rate"] * 100
    )

    return result.sort_values(
        "claim_rate_pct",
        ascending=False,
    )

In [14]:
for feature in [
    "risk_zone",
    "channel",
    "brand",
    "fuel_type",
    "vehicle_usage",
    "previous_claims",
]:
    print(f"\n--- {feature} ---")
    display(categorical_claim_rate(df, feature))


--- risk_zone ---


,category,contracts,claims,claim_rate,claim_rate_pct
1,Low,1286,28,0.0218,2.1773
0,High,1195,26,0.0218,2.1757
2,Medium,1846,34,0.0184,1.8418



--- channel ---


,category,contracts,claims,claim_rate,claim_rate_pct
3,Web,1072,24,0.0224,2.2388
1,Broker,1144,25,0.0219,2.1853
0,Agency,1071,21,0.0196,1.9608
2,Phone,1040,18,0.0173,1.7308



--- brand ---


...

,category,contracts,claims,claim_rate,claim_rate_pct
2,Gasoline,1119,26,0.0232,2.3235
1,Electric,1074,22,0.0205,2.0484
0,Diesel,1053,21,0.0199,1.9943
3,Hybrid,1081,19,0.0176,1.7576



--- vehicle_usage ---


,category,contracts,claims,claim_rate,claim_rate_pct
1,Personal,1441,32,0.0222,2.2207
0,Mixed,1454,29,0.0199,1.9945
2,Professional,1432,27,0.0189,1.8855



--- previous_claims ---


,category,contracts,claims,claim_rate,claim_rate_pct
2,2.0000,937,24,0.0256,2.5614
0,0.0000,965,23,0.0238,2.3834
1,1.0000,1016,18,0.0177,1.7717
3,3.0000,976,16,0.0164,1.6393
4,Missing,433,7,0.0162,1.6166


## Univariate and Bivariate Findings

The numeric features are generally within reasonable ranges.

`current_value` has a strong right-skew. Annual premium and vehicle age have moderate right-skew.

Claim rates change across some categorical variables, but the differences are generally small.

Risk zone does not show a clear ordered relationship with claim occurrence.

Vehicle brand, fuel type, channel, and vehicle usage show some differences in claim rates.

`previous_claims` has a non-linear relationship with the target. Because of this, it may be better to treat this variable as categorical.

In [15]:
def numeric_claim_rate_by_quantile(
    df,
    feature,
    q=5,
):
    temp = df[
        [feature, "has_claim"]
    ].dropna().copy()

    temp["bin"] = pd.qcut(
        temp[feature],
        q=q,
        duplicates="drop",
    )

    result = (
        temp.groupby(
            "bin",
            observed=True,
        )
        .agg(
            contracts=("has_claim", "size"),
            claims=("has_claim", "sum"),
            claim_rate=("has_claim", "mean"),
        )
        .reset_index()
    )

    result["claim_rate_pct"] = (
        result["claim_rate"] * 100
    )

    return result

In [16]:
df["vehicle_age_clean"] = (
    df["vehicle_age"]
    .clip(lower=0)
)

In [17]:
for feature in [
    "annual_premium",
    "client_age",
    "power_hp",
    "current_value",
    "vehicle_age_clean",
]:
    print(f"\n--- {feature} ---")

    display(
        numeric_claim_rate_by_quantile(
            df,
            feature,
        )
    )


--- annual_premium ---


,bin,contracts,claims,claim_rate,claim_rate_pct
0,"(448.29900000000004, 629.978]",866,17,0.0196,1.9630
1,"(629.978, 741.062]",865,17,0.0197,1.9653
2,"(741.062, 878.174]",865,16,0.0185,1.8497
3,"(878.174, 1068.404]",865,16,0.0185,1.8497
4,"(1068.404, 1820.06]",866,22,0.0254,2.5404



--- client_age ---


,bin,contracts,claims,claim_rate,claim_rate_pct
0,"(17.999, 33.0]",843,13,0.0154,1.5421
1,"(33.0, 41.0]",864,24,0.0278,2.7778
2,"(41.0, 47.0]",732,11,0.0150,1.5027
3,"(47.0, 54.0]",769,18,0.0234,2.3407
4,"(54.0, 75.0]",756,13,0.0172,1.7196



--- power_hp ---


,bin,contracts,claims,claim_rate,claim_rate_pct
0,"(88.509, 112.44]",726,12,0.0165,1.6529
1,"(112.44, 134.0]",738,13,0.0176,1.7615
2,"(134.0, 155.56]",708,15,0.0212,2.1186
3,"(155.56, 178.0]",725,15,0.0207,2.0690
4,"(178.0, 200.0]",718,20,0.0279,2.7855



--- current_value ---


,bin,contracts,claims,claim_rate,claim_rate_pct
0,"(4000.0989999999997, 5134.962]",866,13,0.0150,1.5012
1,"(5134.962, 6321.418]",865,14,0.0162,1.6185
2,"(6321.418, 10099.994]",865,25,0.0289,2.8902
3,"(10099.994, 14049.956]",865,20,0.0231,2.3121
4,"(14049.956, 47482.8]",866,16,0.0185,1.8476



--- vehicle_age_clean ---


,bin,contracts,claims,claim_rate,claim_rate_pct
0,"(-0.001, 1.0]",1025,21,0.0205,2.0488
1,"(1.0, 3.0]",839,22,0.0262,2.6222
2,"(3.0, 5.0]",914,14,0.0153,1.5317
3,"(5.0, 8.0]",616,14,0.0227,2.2727
4,"(8.0, 15.0]",716,12,0.0168,1.6760


### Numeric Feature Findings

The numeric features do not show strong linear relationships with claim occurrence.

Vehicle power shows the clearest pattern. Higher-power vehicles generally have a higher claim rate.

Annual premium has a higher claim rate only in the highest premium group.

Client age, vehicle value, and vehicle age show non-linear patterns.

These variables will be kept as candidate features, but more flexible models may be useful for some relationships.

In [18]:
numeric_corr_features = [
    "annual_premium",
    "client_age",
    "power_hp",
    "current_value",
    "previous_claims",
    "vehicle_age_clean",
]

correlation_matrix = (
    df[numeric_corr_features]
    .corr(method="spearman")
    .round(3)
)

correlation_matrix

,annual_premium,client_age,power_hp,current_value,previous_claims,vehicle_age_clean
annual_premium,1.0000,-0.0940,0.0210,0.3490,0.0110,-0.2680
client_age,-0.0940,1.0000,0.0030,0.2440,-0.0240,-0.2730
power_hp,0.0210,0.0030,1.0000,0.0350,-0.0280,-0.0230
current_value,0.3490,0.2440,0.0350,1.0000,0.0250,-0.8470
previous_claims,0.0110,-0.0240,-0.0280,0.0250,1.0000,-0.0130
vehicle_age_clean,-0.2680,-0.2730,-0.0230,-0.8470,-0.0130,1.0000


## Multivariate Findings

Most numeric features have weak correlations with each other.

The strongest relationship is between `current_value` and `vehicle_age`. The Spearman correlation is about -0.85.

This means that older vehicles usually have a lower current value.

However, vehicle age and vehicle value have different business meanings. Both variables will remain as candidate features.

No other strong numeric redundancy was found.

## EDA Conclusions

The final dataset contains 4,327 Auto contracts with vehicle information.

There are 88 contracts with a valid claim, so the target is highly imbalanced.

No single feature strongly explains claim occurrence.

Vehicle power shows the clearest ordered pattern with claim rate. Other variables such as client age, vehicle value, vehicle age, and previous claims show non-linear relationships.

Risk zone does not show a clear ordered relationship with claim occurrence.

Vehicle age and current vehicle value are strongly related, but they represent different business information.

The dataset is ready for feature engineering and model development.

## Feature Engineering Strategy

Based on the EDA:

- Create `vehicle_age` from contract start year and vehicle year.
- Change negative vehicle ages to zero.
- Treat `previous_claims` as a categorical feature.
- Use `Unknown` for missing categorical values.
- Impute missing numeric values during preprocessing.
- Keep the main customer, contract, and vehicle risk features.
- Do not use identifiers or post-claim information as predictors.
- Test vehicle model and color only as optional features.
- Keep both vehicle age and vehicle value as candidate features.

In [19]:
df.head()

,contract_id,start_date,end_date,annual_premium,city,risk_zone,client_age,channel,csp,gender,brand,model,year,power_hp,fuel_type,current_value,color,vehicle_usage,previous_claims,has_claim,vehicle_age,vehicle_age_clean
0,CTR_000005,2025-03-02,2026-02-26,977.5900,Marseille,Medium,39.0000,Web,Employee,Male,Renault,Megane,"2,024.0000",150.0000,Hybrid,"14,873.9200",Black,Personal,0.0000,0,1.0000,1.0000
1,CTR_000012,2024-03-16,2025-04-20,878.3000,Marseille,Medium,41.0000,Phone,Worker,NaN,Peugeot,208,"2,020.0000",175.0000,Hybrid,"8,362.3000",White,Professional,2.0000,0,4.0000,4.0000
2,CTR_000015,2024-06-16,2025-06-14,702.7100,Toulouse,Medium,53.0000,Agency,Employee,Female,Renault,Captur,"2,024.0000",NaN,Electric,"14,849.3000",NaN,Mixed,1.0000,0,0.0000,0.0000
3,CTR_000016,2024-07-18,2025-08-17,967.0000,Lyon,High,47.0000,Agency,Self_employed,Female,Renault,Megane,NaN,176.0000,Gasoline,"14,522.0800",Blue,Personal,1.0000,0,NaN,NaN
4,CTR_000018,2023-09-19,2024-08-05,732.3000,Bordeaux,Medium,36.0000,Web,NaN,NaN,Renault,Megane,"2,018.0000",163.6000,Diesel,"5,563.7300",Blue,Personal,NaN,0,5.0000,5.0000


## Candidate Feature Engineering

The original variables show mostly weak or non-linear relationships with claim occurrence.

For this reason, several new features are created from the existing contract and vehicle information.

The goal is to capture additional risk information without using post-claim variables.

These features are only candidates. They will be kept for modeling only if they show useful patterns during EDA.

In [20]:
feature_df = df.copy()

In [21]:
feature_df["vehicle_age"] = (
    feature_df["start_date"].dt.year
    - feature_df["year"]
).clip(lower=0)

feature_df["contract_duration_days"] = (
    feature_df["end_date"]
    - feature_df["start_date"]
).dt.days

feature_df["premium_value_ratio"] = (
    feature_df["annual_premium"]
    / feature_df["current_value"]
)

feature_df["power_value_ratio"] = (
    feature_df["power_hp"]
    / feature_df["current_value"]
    * 1000
)

feature_df["vehicle_age_group"] = pd.cut(
    feature_df["vehicle_age"],
    bins=[-1, 3, 7, np.inf],
    labels=[
        "0-3",
        "4-7",
        "8+",
    ],
)

feature_df["client_age_group"] = pd.cut(
    feature_df["client_age"],
    bins=[
        17,
        29,
        39,
        49,
        59,
        np.inf,
    ],
    labels=[
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60+",
    ],
)

# missing flags

feature_df["power_missing"] = (
    feature_df["power_hp"]
    .isna()
    .astype(int)
)

feature_df["vehicle_year_missing"] = (
    feature_df["year"]
    .isna()
    .astype(int)
)

feature_df["client_age_missing"] = (
    feature_df["client_age"]
    .isna()
    .astype(int)
)

feature_df["previous_claims_missing"] = (
    feature_df["previous_claims"]
    .isna()
    .astype(int)
)

feature_df["previous_claims_cat"] = (
    feature_df["previous_claims"]
    .astype("Int64")
    .astype("string")
    .fillna("Unknown")
)

In [22]:
candidate_features = [
    "vehicle_age",
    "contract_duration_days",
    "premium_value_ratio",
    "power_value_ratio",
    "vehicle_age_group",
    "client_age_group",
    "power_missing",
    "vehicle_year_missing",
    "client_age_missing",
    "previous_claims_missing",
    "previous_claims_cat",
]

feature_df[candidate_features].head()

,vehicle_age,contract_duration_days,premium_value_ratio,power_value_ratio,vehicle_age_group,client_age_group,power_missing,vehicle_year_missing,client_age_missing,previous_claims_missing,previous_claims_cat
0,1.0000,361,0.0657,10.0848,0-3,30-39,0,0,0,0,0
1,4.0000,400,0.1050,20.9273,4-7,40-49,0,0,0,0,2
2,0.0000,363,0.0473,NaN,0-3,50-59,1,0,0,0,1
3,NaN,395,0.0666,12.1195,NaN,40-49,0,1,0,0,1
4,5.0000,321,0.1316,29.4047,4-7,30-39,0,0,0,1,Unknown


In [23]:
feature_df[
    [
        "vehicle_age",
        "contract_duration_days",
        "premium_value_ratio",
        "power_value_ratio",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
vehicle_age,"4,110.0000",4.5788,3.8153,0.0000,2.0000,4.0000,6.0000,15.0000
contract_duration_days,"4,327.0000",364.4165,25.0031,305.0000,347.0000,364.0000,382.0000,424.0000
premium_value_ratio,"4,327.0000",0.1069,0.0558,0.0260,0.0647,0.0969,0.1319,0.4313
power_value_ratio,"3,615.0000",18.9373,9.8088,1.9694,10.8950,17.3849,25.9628,48.7970


In [24]:
for column in [
    "power_missing",
    "vehicle_year_missing",
    "client_age_missing",
    "previous_claims_missing",
]:
    print(f"\n--- {column} ---")
    print(feature_df[column].value_counts())


--- power_missing ---
power_missing
0    3615
1     712
Name: count, dtype: int64

--- vehicle_year_missing ---
vehicle_year_missing
0    4110
1     217
Name: count, dtype: int64

--- client_age_missing ---
client_age_missing
0    3964
1     363
Name: count, dtype: int64

--- previous_claims_missing ---
previous_claims_missing
0    3894
1     433
Name: count, dtype: int64


### Engineered Feature Analysis

The new features were created successfully.

The next step is to check whether these features show useful differences in claim occurrence.

Numeric features will be analyzed in groups, while categorical and missingness features will be compared using claim rates.

Only useful and interpretable features will be moved to the modeling notebook.

In [29]:
def categorical_claim_rate(df, feature):
    category = (
        df[feature]
        .astype("object")
        .fillna("Missing")
    )

    result = (
        df.assign(category=category)
        .groupby(
            "category",
            observed=True,
        )
        .agg(
            contracts=("contract_id", "size"),
            claims=("has_claim", "sum"),
            claim_rate=("has_claim", "mean"),
        )
        .reset_index()
    )

    result["claim_rate_pct"] = (
        result["claim_rate"] * 100
    )

    return result.sort_values(
        "claim_rate_pct",
        ascending=False,
    )

In [30]:
for feature in [
    "contract_duration_days",
    "premium_value_ratio",
    "power_value_ratio",
]:
    print(f"\n--- {feature} ---")

    display(
        claim_rate_by_quantile(
            feature_df,
            feature,
        )
    )


--- contract_duration_days ---


,bin,contracts,claims,claim_rate,claim_rate_pct
0,"(304.999, 342.0]",880,19,0.0216,2.1591
1,"(342.0, 358.0]",899,19,0.0211,2.1135
2,"(358.0, 371.0]",870,21,0.0241,2.4138
3,"(371.0, 387.0]",855,15,0.0175,1.7544
4,"(387.0, 424.0]",823,14,0.0170,1.7011



--- premium_value_ratio ---


,bin,contracts,claims,claim_rate,claim_rate_pct
0,"(0.024999999999999998, 0.0594]",866,22,0.0254,2.5404
1,"(0.0594, 0.0838]",865,16,0.0185,1.8497
2,"(0.0838, 0.109]",865,23,0.0266,2.6590
3,"(0.109, 0.144]",865,13,0.0150,1.5029
4,"(0.144, 0.431]",866,14,0.0162,1.6166



--- power_value_ratio ---


,bin,contracts,claims,claim_rate,claim_rate_pct
0,"(1.9680000000000002, 9.815]",723,14,0.0194,1.9364
1,"(9.815, 14.429]",723,14,0.0194,1.9364
2,"(14.429, 20.479]",723,21,0.0290,2.9046
3,"(20.479, 28.219]",723,16,0.0221,2.2130
4,"(28.219, 48.797]",723,10,0.0138,1.3831


In [33]:
for feature in [
    "vehicle_age_group",
    "client_age_group",
]:
    print(f"\n--- {feature} ---")

    display(
        categorical_claim_rate(
            feature_df,
            feature,
        )
    )


--- vehicle_age_group ---


,category,contracts,claims,claim_rate,claim_rate_pct
0,0-3,1864,43,0.0231,2.3069
3,Missing,217,5,0.0230,2.3041
1,4-7,1413,27,0.0191,1.9108
2,8+,833,13,0.0156,1.5606



--- client_age_group ---


,category,contracts,claims,claim_rate,claim_rate_pct
5,Missing,363,9,0.0248,2.4793
1,30-39,880,21,0.0239,2.3864
3,50-59,919,19,0.0207,2.0675
2,40-49,1209,24,0.0199,1.9851
0,18-29,571,9,0.0158,1.5762
4,60+,385,6,0.0156,1.5584


In [34]:
display(
    categorical_claim_rate(
        feature_df,
        "previous_claims_cat",
    )
)

,category,contracts,claims,claim_rate,claim_rate_pct
2,2,937,24,0.0256,2.5614
0,0,965,23,0.0238,2.3834
1,1,1016,18,0.0177,1.7717
3,3,976,16,0.0164,1.6393
4,Unknown,433,7,0.0162,1.6166


In [32]:
for feature in [
    "power_missing",
    "vehicle_year_missing",
    "client_age_missing",
    "previous_claims_missing",
]:
    print(f"\n--- {feature} ---")

    result = (
        feature_df
        .groupby(feature)
        .agg(
            contracts=("contract_id", "size"),
            claims=("has_claim", "sum"),
            claim_rate=("has_claim", "mean"),
        )
        .reset_index()
    )

    result["claim_rate_pct"] = (
        result["claim_rate"] * 100
    )

    display(result)


--- power_missing ---


,power_missing,contracts,claims,claim_rate,claim_rate_pct
0,0,3615,75,0.0207,2.0747
1,1,712,13,0.0183,1.8258



--- vehicle_year_missing ---


,vehicle_year_missing,contracts,claims,claim_rate,claim_rate_pct
0,0,4110,83,0.0202,2.0195
1,1,217,5,0.0230,2.3041



--- client_age_missing ---


,client_age_missing,contracts,claims,claim_rate,claim_rate_pct
0,0,3964,79,0.0199,1.9929
1,1,363,9,0.0248,2.4793



--- previous_claims_missing ---


,previous_claims_missing,contracts,claims,claim_rate,claim_rate_pct
0,0,3894,81,0.0208,2.0801
1,1,433,7,0.0162,1.6166


## Engineered Feature Findings

Some engineered features show useful patterns with claim occurrence.

`vehicle_age_group` shows the clearest pattern. Younger vehicles have a higher claim rate than older vehicles.

`client_age_group` also shows differences in claim rates, but the relationship is not linear.

`previous_claims` continues to show a non-linear relationship with claim occurrence. Therefore, its categorical version will be used.

`contract_duration_days` shows a small difference across groups and will be tested during modeling.

`premium_value_ratio` shows some differences, but the pattern is not stable. It will be treated as an optional feature.

`power_value_ratio` does not show a clear pattern and will not be included in the main feature set.

Separate missing indicators are not needed when the same missing information is already represented by categorical features.

## Final Feature Engineering Strategy

The modeling stage will compare two feature sets.

### Core Feature Set

The core set contains the original contract, customer, and vehicle risk variables.

### Extended Feature Set

The extended set adds selected engineered features:

- `vehicle_age_group`
- `client_age_group`
- `contract_duration_days`
- `premium_value_ratio`

`previous_claims` will be represented as a categorical variable.

The engineered features will only be kept in the final model if they improve repeated cross-validation performance.

Post-claim information will not be used.